In [54]:
import pandas as pd
from pathlib import Path
def read_data(path: str):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{path}: not found')
    ext = path.suffix
    if ext == '.csv':
        return pd.read_csv(path)
    elif ext in set(['.xls', '.xlsx']):
        return pd.read_excel(path)
    raise ValueError(f'File extension: {ext}; not supported')

# df = read_data('data/nlp_expanded.xlsx')
df = read_data('data/nlp.xlsx')

In [58]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

def extract_absolute_recall_keywords(
    unlabeled_data, 
    target_labels,
    text_column='Discrepancy', 
    top_n=15, 
    min_global_df=5, 
    output_path='final_unsupervised_dictionary.json'
):
    """
    Production-grade Zero-Shot Keyword extractor built for absolute recall.
    Bypasses POS tagger failure modes by using deterministic substring matching 
    to force critical engineering unigrams (like corrosion) into the output.
    """
    print("Initializing embedding space on CPU...")
    model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    
    if isinstance(unlabeled_data, list):
        df = pd.DataFrame(unlabeled_data)
    else:
        df = unlabeled_data.copy()
        
    df = df.dropna(subset=[text_column])
    corpus = df[text_column].astype(str).str.lower().tolist()
    
    # Clean text columns from heavy punctuation or numbers that might confuse vocabulary splits
    print("Normalizing corpus tokens...")
    normalized_corpus = []
    for text in corpus:
        # Retain characters, spaces, and hyphens
        cleaned = " ".join(word for word in text.split() if len(word) >= 3)
        normalized_corpus.append(cleaned)

    print(f"Extracting vocab using TfidfVectorizer (min_df={min_global_df})...")
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),  
        stop_words='english',
        min_df=min_global_df
    )
    
    tfidf_matrix = vectorizer.fit_transform(normalized_corpus)
    feature_names = vectorizer.get_feature_names_out()
    
    df_counts = np.bincount(tfidf_matrix.indices, minlength=tfidf_matrix.shape[1])
    total_docs = tfidf_matrix.shape[0]
    num_features = len(feature_names)
    print(f"Total candidate vocabulary space: {num_features} unique tokens/phrases.")
    
    print("Pre-encoding category profiles...")
    label_embeddings = {
        label.lower(): model.encode(label.lower(), convert_to_tensor=False) 
        for label in target_labels
    }
    
    print("Embedding candidate vocabulary variants...")
    feature_embeddings = model.encode(list(feature_names), convert_to_tensor=False, batch_size=1024, show_progress_bar=True)
    
    final_output_dictionary = {}
    
    for label_str, target_embed in label_embeddings.items():
        print(f"Computing discriminative alignments for label concept: '{label_str}'...")
        target_vector = target_embed.reshape(1, -1)
        semantic_scores = cosine_similarity(target_vector, feature_embeddings)[0]
        
        # We define a strict 4-letter core root boundary for morphological protection
        root_stem = label_str[:4] # 'corr', 'crac', 'dela', 'leak'
        
        label_results = []
        for idx, token in enumerate(feature_names):
            base_semantic = semantic_scores[idx]
            token_words = token.split()
            
            # ABSOLUTE RECALL TRAP: If a unigram starts with our core stem, 
            # we manually max out its relatedness score so it cannot be ignored or outcompeted.
            is_core_root = False
            if len(token_words) == 1 and token.startswith(root_stem):
                is_core_root = True
                
            prefix_boost = 0.0
            for w in token_words:
                if w.startswith(root_stem) or root_stem.startswith(w[:4]):
                    prefix_boost = 0.50
                    
            relatedness_score = 1.5 if is_core_root else (base_semantic + prefix_boost)
            
            doc_popularity = df_counts[idx] / total_docs
            specificity_penalty = 1.0 - doc_popularity
            
            final_score = relatedness_score * specificity_penalty
            
            # Allow safe padding for semantic matches (like "fractured" matching "cracked")
            if is_core_root or relatedness_score > 0.38:
                label_results.append((token, final_score, is_core_root))
                
        # Sort features: Priority 1 is whether it's a core root unigram, Priority 2 is final score weight
        label_results = sorted(label_results, key=lambda x: (x[2], x[1]), reverse=True)
        
        clean_keywords = []
        for token, _, is_core_root in label_results:
            token_words = token.split()
            
            # Unigrams take absolute priority
            if len(token_words) == 1:
                if token not in clean_keywords:
                    clean_keywords.append(token)
            else:
                # Strip out loose action bigram combinations
                if any(w in clean_keywords for w in token_words):
                    if any(w in ["noted", "showed", "added", "worked", "found", "checking"] for w in token_words):
                        continue
                if token not in clean_keywords:
                    clean_keywords.append(token)
                    
            if len(clean_keywords) >= top_n:
                break
                
        final_output_dictionary[label_str] = clean_keywords
        
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(final_output_dictionary, f, indent=4, ensure_ascii=False)
        
    print(f"\nPipeline Execution Complete! File saved to: '{output_path}'")
    return final_output_dictionary

# =====================================================================
# Pipeline Execution Entry Point
# =====================================================================
if __name__ == "__main__":
    my_target_labels = ["corroded", "cracked", "delaminated", "leaking", "odor", "dirty", "loose", "worn"]
    
    resulting_json = extract_absolute_recall_keywords(
        unlabeled_data=df,  # Your data frame variable
        target_labels=my_target_labels,
        text_column='Discrepancy',
        top_n=15,
        min_global_df=15,    # Increase to 10 or 15 for your 170k row variant
        output_path='final_unsupervised_dictionary.json'
    )

Initializing embedding space on CPU...
Normalizing corpus tokens...
Extracting vocab using TfidfVectorizer (min_df=15)...
Total candidate vocabulary space: 2107 unique tokens/phrases.
Pre-encoding category profiles...
Embedding candidate vocabulary variants...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Computing discriminative alignments for label concept: 'corroded'...
Computing discriminative alignments for label concept: 'cracked'...
Computing discriminative alignments for label concept: 'delaminated'...
Computing discriminative alignments for label concept: 'leaking'...
Computing discriminative alignments for label concept: 'odor'...
Computing discriminative alignments for label concept: 'dirty'...
Computing discriminative alignments for label concept: 'loose'...
Computing discriminative alignments for label concept: 'worn'...

Pipeline Execution Complete! File saved to: 'final_unsupervised_dictionary.json'
